<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI LangGraph Agent Tutorial: Hello World — Backbone Sweep

This notebook demonstrates how to evaluate LangGraph agents using RapidFire AI.
We'll sweep multiple LLM backbones on a simple Q&A task using a prebuilt ReAct agent,
compare their accuracy and cost side-by-side, and show how the same workflow extends
to custom multi-node graphs.

**What you'll learn:**
- How to use `RFLangGraphAgentConfig` with prebuilt agents
- How to sweep model backbones with `List()`
- How to define `input_mapper`, `output_mapper`, metrics functions
- How to run hyperparallelized agent evaluation with `run_evals()`
- How to use custom graph factories for architecture sweeps

### Prerequisites

```bash
pip install rapidfireai langgraph langchain-openai
```

Set your OpenAI API key:
```bash
export OPENAI_API_KEY="sk-..."
```

In [ ]:
import os

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFLangGraphAgentConfig

### 1. Create a Simple Q&A Dataset

We'll use a tiny inline dataset with questions and expected answers.
In production, replace this with GAIA, HotpotQA, or your own eval set.

In [ ]:
from datasets import Dataset

qa_data = {
    "question": [
        "What is the capital of France?",
        "What is 25 * 4?",
        "Who wrote Romeo and Juliet?",
        "What is the boiling point of water in Celsius?",
        "What planet is closest to the Sun?",
        "What is the square root of 144?",
        "In which year did World War II end?",
        "What is the chemical symbol for gold?",
    ],
    "expected_answer": [
        "Paris",
        "100",
        "Shakespeare",
        "100",
        "Mercury",
        "12",
        "1945",
        "Au",
    ],
}

dataset = Dataset.from_dict(qa_data)
print(f"Dataset size: {len(dataset)} questions")

### 2. Create Experiment

In [ ]:
experiment = Experiment(experiment_name="langgraph-hello-world", mode="evals")

### 3. Define Tools (Optional)

For this hello-world example we use a simple calculator tool.
The ReAct agent will decide whether to use it based on the question.

In [ ]:
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


my_tools = [calculator]

### 4. Define Input/Output Mappers and Metrics

- `input_mapper`: converts a dataset row → graph input dict
- `output_mapper`: converts graph result state → answer string
- `compute_metrics_fn`: per-batch accuracy
- `accumulate_metrics_fn`: weighted aggregation across batches

In [ ]:
from langchain_core.messages import HumanMessage


def input_mapper(row):
    """Convert a dataset row to the graph's input format."""
    return {"messages": [HumanMessage(content=row["question"])]}


def output_mapper(result):
    """Extract the final answer from the graph's output state."""
    if result is None:
        return ""
    messages = result.get("messages", [])
    if messages:
        return messages[-1].content
    return ""


def compute_metrics_fn(batch):
    """Compute per-batch accuracy (fuzzy match)."""
    outputs = batch.get("agent_output", [])
    expected = batch.get("expected_answer", [])

    correct = 0
    for pred, gt in zip(outputs, expected):
        if gt.lower() in pred.lower():
            correct += 1

    n = len(outputs)
    return {
        "accuracy": {"value": correct / n if n > 0 else 0, "is_algebraic": True},
    }


def accumulate_metrics_fn(metrics):
    """Weighted-average accumulator across batches."""
    accuracy_batches = metrics.get("accuracy", [])
    sample_batches = metrics.get("Samples Processed", [])

    total_samples = sum(
        (s.get("value", 1) if isinstance(s, dict) else s) for s in sample_batches
    )
    weighted_acc = sum(
        (a.get("value", 0) if isinstance(a, dict) else a)
        * (s.get("value", 1) if isinstance(s, dict) else s)
        for a, s in zip(accuracy_batches, sample_batches)
    )

    return {
        "accuracy": {
            "value": round(weighted_acc / total_samples, 4) if total_samples > 0 else 0,
            "is_algebraic": True,
        },
    }

### 5. Configure the Agent Sweep

We use `RFLangGraphAgentConfig` with `prebuilt_agent="react"` and sweep
across multiple OpenAI models. RF will create one pipeline per model,
shard the dataset, and evaluate them hyperparallelized.

In [ ]:
from langchain_openai import ChatOpenAI

config = RFGridSearch(
    {
        "pipeline": RFLangGraphAgentConfig(
            prebuilt_agent="react",
            prebuilt_agent_kwargs={
                "model": List(
                    [
                        ChatOpenAI(model="gpt-4o-mini", temperature=0),
                        ChatOpenAI(model="gpt-4o", temperature=0),
                    ]
                ),
                "tools": my_tools,
            },
            input_mapper=input_mapper,
            output_mapper=output_mapper,
            max_steps=10,
            timeout_seconds=60,
        ),
        "batch_size": 1,
        "compute_metrics_fn": compute_metrics_fn,
        "accumulate_metrics_fn": accumulate_metrics_fn,
    }
)

# This will produce 2 pipeline configs (one per model)

### 6. Run Evaluation

- `num_shards`: many small shards (agents have variable latency)
- `num_actors`: limited by API rate limits, not CPU count
- `batch_size=1`: agent invocations are sequential per row

In [ ]:
results = experiment.run_evals(
    config_group=config,
    dataset=dataset,
    num_shards=4,
    seed=42,
)

### 7. View Results

In [ ]:
import pandas as pd

rows = []
for pipeline_id, (aggregated_results, cumulative_metrics) in results.items():
    row = {"pipeline_id": pipeline_id}
    for metric_name, metric_data in cumulative_metrics.items():
        if isinstance(metric_data, dict):
            row[metric_name] = metric_data.get("value", metric_data)
        else:
            row[metric_name] = metric_data
    rows.append(row)

df = pd.DataFrame(rows)
df

### 8. (Optional) Custom Graph Factory

For more control, define a `graph_fn` that builds an **uncompiled**
`StateGraph`. RF will compile it. Every argument to the factory
can be swept with `List()` or `Range()`.

In [ ]:
from langgraph.graph import StateGraph, MessagesState


def make_simple_qa_graph(model_name="gpt-4o-mini", temperature=0.0):
    """Factory that returns an UNCOMPILED StateGraph builder."""
    llm = ChatOpenAI(model=model_name, temperature=temperature)

    def answer_node(state: MessagesState):
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    builder = StateGraph(MessagesState)
    builder.add_node("answer", answer_node)
    builder.set_entry_point("answer")
    builder.set_finish_point("answer")

    return builder  # Uncompiled — RF compiles


# Architecture sweep: compare prebuilt ReAct vs. simple chain
architecture_config = RFGridSearch(
    {
        "pipeline": List(
            [
                # Simple single-node chain (no tools)
                RFLangGraphAgentConfig(
                    graph_fn=make_simple_qa_graph,
                    model_name=List(["gpt-4o-mini", "gpt-4o"]),
                    temperature=0.0,
                    input_mapper=input_mapper,
                    output_mapper=output_mapper,
                    max_steps=5,
                ),
                # ReAct agent with tools
                RFLangGraphAgentConfig(
                    prebuilt_agent="react",
                    prebuilt_agent_kwargs={
                        "model": ChatOpenAI(model="gpt-4o-mini", temperature=0),
                        "tools": my_tools,
                    },
                    input_mapper=input_mapper,
                    output_mapper=output_mapper,
                    max_steps=10,
                ),
            ]
        ),
        "batch_size": 1,
        "compute_metrics_fn": compute_metrics_fn,
        "accumulate_metrics_fn": accumulate_metrics_fn,
    }
)

# 2 simple-chain models + 1 ReAct = 3 configs total
print("Architecture sweep configured — ready to run with experiment.run_evals()")

### 9. End Experiment

In [ ]:
experiment.end()